# Foto → humor e energia — Parte 1: treinar o modelo

**O que este notebook faz:** ensina um modelo a olhar uma foto e dar duas notas:
**valência** (humor: desagradável → agradável) e **energia** (arousal: calma → agitada).

**Como:**
1. O **CLIP** (ViT-B/32), já pré-treinado, transforma cada foto num vetor de 512 números (*embedding*). Ele fica **congelado** — não mexemos nos pesos dele.
2. Uma **regressão Ridge de 2 saídas** aprende a ir do vetor para as notas, usando as **900 fotos do OASIS**, que foram avaliadas por ~100 pessoas cada (escala 1–7).
3. Avaliamos com **validação cruzada agrupada** e, no fim, treinamos com tudo e salvamos o modelo no Drive.

No teste de viabilidade (14/11/2026) esse método deu **r = 0,85 (humor)** e **r = 0,82 (energia)**. Este notebook deve reproduzir algo bem próximo.

> Ambiente: Colab com GPU é mais rápido (Ambiente de execução → Alterar tipo → T4), mas na CPU também roda em poucos minutos.

## 1. Configuração
Ajuste `PASTA` para a pasta do projeto no seu Drive.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = Path("/content/drive/MyDrive/mapa-sonoro")   # <- ajuste se a sua pasta tiver outro nome
    NO_COLAB = True
except ImportError:                                        # rodando fora do Colab
    PASTA = Path("mapa-sonoro-local")
    NO_COLAB = False

PASTA_OASIS = PASTA / "oasis"
PASTA_MODELOS = PASTA / "modelos"
PASTA_OASIS.mkdir(parents=True, exist_ok=True)
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

SEMENTE = 0
NOME_CLIP = "openai/clip-vit-base-patch32"
print("Pasta do projeto:", PASTA)

## 2. Baixar o OASIS
Só baixa na primeira vez (~90 MB); depois fica guardado no Drive.
O OASIS é de uso livre em pesquisa (Kurdi, Lozano & Banaji, 2017).

In [ ]:
import subprocess, zipfile

csv_oasis = PASTA_OASIS / "OASIS.csv"
if not csv_oasis.exists():
    zip_path = PASTA_OASIS / "oasis.zip"
    subprocess.run(["curl", "-sSL", "-o", str(zip_path), "https://benedekkurdi.com/oasis.php"], check=True)
    with zipfile.ZipFile(zip_path) as z:
        for membro in z.namelist():
            if membro.startswith("__MACOSX") or membro.endswith(".DS_Store"):
                continue                    # lixo do macOS dentro do zip
            z.extract(membro, PASTA_OASIS)
    zip_path.unlink()

print("Imagens:", len(list((PASTA_OASIS / "Images").glob("*.jpg"))))

## 3. Ler as notas
Cada linha é uma foto. Usamos a **média** das notas (`Valence_mean`, `Arousal_mean`).

**Grupo:** o OASIS tem séries de fotos quase iguais ("Acorns 1", "Acorns 2", "Acorns 3").
Se uma ficar no treino e a irmã no teste, o modelo "cola" e o resultado fica otimista demais.
Por isso criamos um **grupo por tema**, e fotos do mesmo grupo ficam sempre do mesmo lado.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(csv_oasis)
df["arquivo"] = df["Theme"].str.strip() + ".jpg"
df["grupo"] = df["Theme"].str.replace(r"\s*\d+$", "", regex=True).str.strip()

Y = df[["Valence_mean", "Arousal_mean"]].to_numpy()
EIXOS = ["valência", "energia"]

print(f"{len(df)} fotos, {df.grupo.nunique()} temas")
print(df.Category.value_counts().to_string())
print(f"\nCorrelação entre valência e energia nas notas humanas: {np.corrcoef(Y.T)[0, 1]:.2f}"
      "  (perto de 0 = eixos independentes, bom para usar os dois)")
df[["Theme", "Category", "Valence_mean", "Arousal_mean"]].describe().round(2)

### Teto de ruído
Notas humanas têm ruído. Com ~100 pessoas por foto, a média é bem confiável.
O **teto** é a correlação máxima que um modelo perfeito conseguiria, dado esse ruído.

In [ ]:
for eixo, col in [("valência", "Valence"), ("energia", "Arousal")]:
    erro_da_media = (df[f"{col}_SD"] ** 2 / df[f"{col}_N"]).mean()
    confiabilidade = 1 - erro_da_media / df[f"{col}_mean"].var()
    print(f"{eixo:9s}  confiabilidade = {confiabilidade:.3f}   teto de r ≈ {np.sqrt(confiabilidade):.3f}")

## 4. Extrair os embeddings com o CLIP
Cada foto vira um vetor de 512 números. Normalizamos o vetor (comprimento 1), como o CLIP faz internamente.
O resultado é salvo no Drive para não precisar repetir.

In [ ]:
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

disp = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained(NOME_CLIP).eval().to(disp)
proc = CLIPProcessor.from_pretrained(NOME_CLIP)

@torch.no_grad()
def embeddings(imagens, lote=32):
    # Recebe uma lista de imagens PIL e devolve uma matriz (n, 512) normalizada
    saidas = []
    for i in range(0, len(imagens), lote):
        x = proc(images=imagens[i:i + lote], return_tensors="pt").to(disp)
        saida_visual = clip.vision_model(pixel_values=x["pixel_values"])
        e = clip.visual_projection(saida_visual.pooler_output)
        saidas.append(torch.nn.functional.normalize(e, dim=-1).cpu())
    return torch.cat(saidas).numpy()

cache = PASTA_MODELOS / "clip_oasis_embeddings.npy"
if cache.exists():
    X = np.load(cache)
else:
    imgs = [Image.open(PASTA_OASIS / "Images" / a).convert("RGB") for a in df.arquivo]
    X = embeddings(imgs)
    np.save(cache, X)
print("X:", X.shape, "| dispositivo:", disp)

## 5. Validação cruzada agrupada (5 folds)
Para cada fold: treina com 80% dos temas e testa nos 20% restantes, que o modelo nunca viu.

O modelo é um `Pipeline`:
- `StandardScaler`: coloca as 512 colunas na mesma escala;
- `RidgeCV`: regressão linear com penalidade (evita decorar com só 900 exemplos). Ele mesmo escolhe a força da penalidade (`alpha`).

Comparamos com a **linha de base**: sempre chutar a média do treino.

In [ ]:
from scipy.stats import pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def novo_modelo():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 20)))

folds = list(GroupKFold(n_splits=5).split(X, Y, groups=df.grupo))
P = np.zeros_like(Y)       # previsões fora da amostra
P0 = np.zeros_like(Y)      # linha de base
alphas = []
for treino, teste in folds:
    m = novo_modelo().fit(X[treino], Y[treino])
    P[teste] = m.predict(X[teste])
    P0[teste] = Y[treino].mean(axis=0)
    alphas.append(m[-1].alpha_)

linhas = []
for j, eixo in enumerate(EIXOS):
    linhas.append({"eixo": eixo,
                   "r": pearsonr(Y[:, j], P[:, j])[0],
                   "erro médio (MAE)": mean_absolute_error(Y[:, j], P[:, j]),
                   "MAE chutando a média": mean_absolute_error(Y[:, j], P0[:, j])})
print("alphas escolhidos por fold:", np.round(alphas, 1))
pd.DataFrame(linhas).round(3)

### Por tipo de foto
`Scene` (paisagens/cenários) e `Person` (retratos) são os casos de uso do projeto.

In [ ]:
por_cat = []
for cat, g in df.groupby("Category"):
    i = g.index
    por_cat.append({"categoria": cat, "n": len(i),
                    "r valência": pearsonr(Y[i, 0], P[i, 0])[0],
                    "r energia": pearsonr(Y[i, 1], P[i, 1])[0]})
pd.DataFrame(por_cat).round(2)

### Previsto × real
Cada ponto é uma foto prevista por um modelo que **não** a viu no treino. Quanto mais perto da diagonal, melhor.

In [ ]:
import matplotlib.pyplot as plt

fig, eixos_plot = plt.subplots(1, 2, figsize=(10, 4.5))
for j, (ax, eixo) in enumerate(zip(eixos_plot, EIXOS)):
    ax.scatter(Y[:, j], P[:, j], s=10, alpha=0.5)
    ax.plot([1, 7], [1, 7], color="gray", lw=1)
    ax.set(xlabel=f"{eixo} real (humanos)", ylabel=f"{eixo} prevista",
           title=f"{eixo}: r = {pearsonr(Y[:, j], P[:, j])[0]:.2f}", xlim=(1, 7), ylim=(1, 7))
plt.tight_layout(); plt.show()

### Onde o modelo mais erra
Útil para entender limites: em geral são fotos cujo sentido depende de contexto (ironia, detalhe pequeno).

In [ ]:
erros = df[["Theme", "Category"]].assign(
    val_real=Y[:, 0], val_prev=P[:, 0], ene_real=Y[:, 1], ene_prev=P[:, 1],
    erro_total=np.abs(Y - P).sum(axis=1))
erros.sort_values("erro_total", ascending=False).head(10).round(2)

## 5b. Fotos do dia a dia: Emotion6 (review, seção 1)

O review (`docs/REVIEW.md`, *Medium: the photo model learns from a narrow source*): *"900 OASIS images, much of it extreme content. [...] Training on everyday photos as well (EMOTIC, FI or Emotion6) matters more than tuning Ridge."*

Das três, só a **Emotion6** tem notas de valência e energia da foto inteira, como o OASIS: 1.980 fotos do Flickr, 15 pessoas por foto, escala SAM de 9 pontos (Peng et al., CVPR 2015). A FI tem só categorias; a EMOTIC anota a emoção aparente de cada pessoa na foto.

**Antes de rodar:** envie o `Emotion6.zip` para a pasta `emotion6/` dentro da pasta do projeto no Drive (`PASTA`, seção 1). A célula descompacta na primeira vez.
Licença (readme da base): só para pesquisa acadêmica; os direitos das imagens são dos donos. **Nem as fotos nem as notas vão para o GitHub.**

**O que mede** (os mesmos 5 folds do OASIS da seção 5; a Emotion6 em 5 folds sorteados, semente `SEMENTE`):
1. **O problema:** o modelo atual (treinado só no OASIS) prevendo a Emotion6, fotos que ele nunca viu.
2. **A correção:** um modelo treinado com OASIS + Emotion6. Cada base entra na **própria escala** (z-score com média e desvio do treino de cada fold), porque as escalas são diferentes (1–7 e 1–9).
3. Medida: r de Pearson (a mesma da seção 5), em cada base separada. Diferença combinado − atual com IC 95% por bootstrap pareado (10.000 reamostras; no OASIS, reamostrando **temas**, como na validação agrupada).

**Regra de decisão (fixada antes de rodar):** o modelo combinado substitui o atual se
- na **Emotion6**, o IC 95% da diferença de r ficar inteiro acima de zero em pelo menos um eixo, sem ficar inteiro abaixo de zero no outro; **e**
- no **OASIS**, o IC da diferença não ficar inteiro abaixo de zero em nenhum eixo (não piorar onde já funciona).

Se substituir, a seção 6 passa a treinar o modelo final com as duas bases (mudança a fazer depois, só nesse caso).
Observação: na Emotion6 a energia varia pouco (desvio 0,88 na escala 1–9; metade das fotos entre 4,5 e 5,6), o que tende a baixar o r de energia em qualquer modelo.


In [ ]:
# 5b — Emotion6: o modelo atual em fotos do dia a dia, e o modelo treinado com as duas bases
import zipfile
from sklearn.model_selection import KFold

# 1) dados
PASTA_E6 = PASTA / "emotion6"
raiz_e6 = PASTA_E6 / "Emotion6"
if not (raiz_e6 / "ground_truth.txt").exists():
    zip_e6 = PASTA_E6 / "Emotion6.zip"
    if not zip_e6.exists():
        raise FileNotFoundError(f"Envie o Emotion6.zip para {PASTA_E6} no Drive (ver o texto acima).")
    with zipfile.ZipFile(zip_e6) as z:
        for membro in z.namelist():
            if membro.startswith("__MACOSX") or membro.endswith(".DS_Store"):
                continue                    # lixo do macOS dentro do zip
            z.extract(membro, PASTA_E6)

e6 = pd.read_csv(raiz_e6 / "ground_truth.txt", sep="\t")
e6.columns = [c.strip("[] ") for c in e6.columns]
Y6 = e6[["valence", "arousal"]].to_numpy()
print(f"Emotion6: {len(e6)} fotos | valência {Y6[:, 0].min()}–{Y6[:, 0].max()} | energia {Y6[:, 1].min()}–{Y6[:, 1].max()}")

# 2) embeddings CLIP (a mesma função da seção 4), guardados no Drive
cache6 = PASTA_MODELOS / "clip_emotion6_embeddings.npy"
if cache6.exists():
    X6 = np.load(cache6)
else:
    X6 = np.concatenate([
        embeddings([Image.open(raiz_e6 / "images" / f).convert("RGB") for f in e6.image_filename[i:i + 200]])
        for i in range(0, len(e6), 200)])
    np.save(cache6, X6)
if len(X6) != len(e6):
    raise ValueError("Embeddings da Emotion6 não batem com o ground_truth.txt: apague o cache e rode de novo.")

# 3) validação cruzada: os mesmos 5 folds do OASIS (seção 5) + 5 folds sorteados na Emotion6
folds6 = list(KFold(n_splits=5, shuffle=True, random_state=SEMENTE).split(X6))
P_atual_e6 = np.zeros_like(Y6)          # modelo atual (só OASIS) prevendo a Emotion6
P_comb_e6 = np.zeros_like(Y6)           # modelo combinado prevendo a Emotion6
P_comb_oasis = np.zeros_like(Y)         # modelo combinado prevendo o OASIS
for (tr, te), (tr6, te6) in zip(folds, folds6):
    atual = novo_modelo().fit(X[tr], Y[tr])                # igual à seção 5
    P_atual_e6[te6] = atual.predict(X6[te6])
    mo, so = Y[tr].mean(axis=0), Y[tr].std(axis=0)         # cada base na própria escala, com média e desvio do treino
    m6, s6 = Y6[tr6].mean(axis=0), Y6[tr6].std(axis=0)
    comb = novo_modelo().fit(np.vstack([X[tr], X6[tr6]]),
                             np.vstack([(Y[tr] - mo) / so, (Y6[tr6] - m6) / s6]))
    P_comb_oasis[te] = comb.predict(X[te])
    P_comb_e6[te6] = comb.predict(X6[te6])
# no OASIS, o modelo atual é o P da seção 5 (mesmos folds)

# 4) r e IC 95% da diferença (bootstrap pareado; no OASIS reamostra temas)
def r(y, p):
    return np.corrcoef(y, p)[0, 1]

def ic_diferenca(y, p_atual, p_novo, grupos=None, n=10_000):
    rng = np.random.default_rng(SEMENTE)
    if grupos is None:
        sorteio = lambda: rng.integers(len(y), size=len(y))
    else:
        membros = [np.flatnonzero(grupos == g) for g in np.unique(grupos)]
        sorteio = lambda: np.concatenate([membros[k] for k in rng.integers(len(membros), size=len(membros))])
    difs = []
    for _ in range(n):
        i = sorteio()
        difs.append(r(y[i], p_novo[i]) - r(y[i], p_atual[i]))
    return np.quantile(difs, [0.025, 0.975])

grupos_oasis = df.grupo.to_numpy()
linhas = []
for base, y, pa, pn, g in (("Emotion6", Y6, P_atual_e6, P_comb_e6, None),
                           ("OASIS", Y, P, P_comb_oasis, grupos_oasis)):
    for j, eixo in enumerate(EIXOS):
        lo, hi = ic_diferenca(y[:, j], pa[:, j], pn[:, j], g)
        linhas.append({"base": base, "eixo": eixo, "r atual (só OASIS)": r(y[:, j], pa[:, j]),
                       "r combinado": r(y[:, j], pn[:, j]), "diferença": r(y[:, j], pn[:, j]) - r(y[:, j], pa[:, j]),
                       "IC 95% inf": lo, "IC 95% sup": hi})
res6 = pd.DataFrame(linhas)
res6.to_csv(PASTA_MODELOS / "emotion6_resultado.csv", index=False)
print("conferência: no OASIS, 'r atual' deve repetir a seção 5\n")
print(res6.round(3).to_string(index=False))

# 5) regra de decisão (fixada antes de rodar)
def eixos(base, cond):
    t = res6[res6.base == base]
    return [e for e, lo, hi in zip(t.eixo, t["IC 95% inf"], t["IC 95% sup"]) if cond(lo, hi)]
melhora_e6 = eixos("Emotion6", lambda lo, hi: lo > 0)
piora_e6 = eixos("Emotion6", lambda lo, hi: hi < 0)
piora_oasis = eixos("OASIS", lambda lo, hi: hi < 0)
if melhora_e6 and not piora_e6 and not piora_oasis:
    print(f"\nDECISÃO: o modelo combinado substitui o atual (melhor na Emotion6 em {', '.join(melhora_e6)}; sem piora).")
else:
    motivo = ([f"piora na Emotion6 em {', '.join(piora_e6)}"] if piora_e6 else []) + \
             ([f"piora no OASIS em {', '.join(piora_oasis)}"] if piora_oasis else []) + \
             ([] if melhora_e6 else ["sem melhora clara na Emotion6"])
    print("\nDECISÃO: o modelo atual continua (" + "; ".join(motivo) + ").")


## 6. Treinar o modelo final com as 900 fotos e salvar
A validação mostrou quanto o método acerta. Agora treinamos com **tudo** (mais dados = modelo um pouco melhor) e salvamos no Drive.

Junto com o modelo, salvamos a **média e o desvio padrão** das notas do OASIS.
Na Parte 2 eles servem para padronizar as notas das fotos (z-score) e compará-las com as das músicas, que usam outra escala (1–9).

In [ ]:
import joblib, sklearn, transformers
from datetime import date

final = novo_modelo().fit(X, Y)

pacote = {
    "modelo": final,
    "clip": NOME_CLIP,
    "eixos": ["valencia", "energia"],
    "escala": "OASIS 1-7",
    "media_oasis": Y.mean(axis=0),
    "desvio_oasis": Y.std(axis=0),
    "validacao": {"r_valencia": float(pearsonr(Y[:, 0], P[:, 0])[0]),
                  "r_energia": float(pearsonr(Y[:, 1], P[:, 1])[0])},
    "versoes": {"sklearn": sklearn.__version__, "transformers": transformers.__version__},
    "data": str(date.today()),
}
arquivo_modelo = PASTA_MODELOS / "foto_humor_energia.joblib"
pacote["media_prev"] = P.mean(axis=0)
pacote["desvio_prev"] = P.std(axis=0)
joblib.dump(pacote, arquivo_modelo)
print("Salvo em", arquivo_modelo, "| alpha final:", round(final[-1].alpha_, 1))

## 7. Testar com uma foto sua
No Colab, rode a célula e escolha uma ou mais fotos. Lembrete honesto: o OASIS tem muitas fotos de conteúdo forte;
fotos do dia a dia tendem a cair numa faixa mais estreita de notas.

In [ ]:
def notas_da_foto(imagens):
    # Devolve uma tabela com valência e energia (escala 1-7) para uma lista de imagens PIL
    pac = joblib.load(arquivo_modelo)
    p = pac["modelo"].predict(embeddings(imagens))
    return pd.DataFrame(p, columns=["valência (1-7)", "energia (1-7)"]).round(2)

if NO_COLAB:
    from google.colab import files
    import io
    enviadas = files.upload()
    nomes = list(enviadas)
    fotos = [Image.open(io.BytesIO(enviadas[n])).convert("RGB") for n in nomes]
else:                                   # fora do Colab: testa com 3 fotos do próprio OASIS
    nomes = df.arquivo[:3].tolist()
    fotos = [Image.open(PASTA_OASIS / "Images" / n).convert("RGB") for n in nomes]

resultado = notas_da_foto(fotos)
resultado.insert(0, "foto", nomes)
resultado